This notebook is based on [LMSYS: KerasNLP Starter](https://www.kaggle.com/code/addisonhoward/lmsys-kerasnlp-starter). Some changes are:

* Handling for `null` in data.
* Handling for characters in data that cannot be represented as utf-8.
* Amout of data used for the training.
* For the model, take the first dimension of the embeddings for this classification task, instead of using GlobalAveragePooling1D in the original model.

# Import Libraries

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import keras_nlp
import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Library Version

In [ ]:
print(f'Keras: {keras.__version__}')
print(f'KerasNLP: {keras_nlp.__version__}')
print(f'Tensorflow: {tf.__version__}')

When debugging, epochs and the amount of training/validation data is reduced to speed up the process.

In [ ]:
DEBUG = False

# Configuration

In [ ]:
class CFG:
    seed = 42
    preset = "deberta_v3_extra_small_en"
    preprocessor = keras_nlp.models.DebertaV3Preprocessor
    backbone = keras_nlp.models.DebertaV3Backbone
    sequence_length = 512
    epochs = 4 if DEBUG else 10
    batch_size = 16
    label2name = {
        0: 'winner_model_a', 1: 'winner_model_b', 2: 'winner_tie'
    }
    name2label = {v: k for k, v in label2name.items()}
    class_labels = list(label2name.keys())
    class_names = list(label2name.values())

print(f'label2name:   {CFG.label2name}')
print(f'name2label:   {CFG.name2label}')
print(f'class_labels: {CFG.class_labels}')
print(f'class_names:  {CFG.class_names}')

# Reproducibility

To produce similar result in each run.

In [ ]:
keras.utils.set_random_seed(CFG.seed)

print(f'Set random seed to {CFG.seed}.')

# Mixed Precision

To reduce GPU memory usage.

In [ ]:
keras.mixed_precision.set_global_policy("mixed_float16")

print(f'Set mixed precision global policy to "mixed_float16".')

# Dataset Path

In [ ]:
BASE_PATH = os.path.join(
    '/', 'kaggle', 'input', 'llm-classification-finetuning')

print(f'BASE_PATH = {BASE_PATH}')
os.listdir(BASE_PATH)

## Training Data

Load training data. 

In [ ]:
train_path = os.path.join(BASE_PATH, 'train.csv')
train_df = pd.read_csv(train_path)

train_df

## Test Data

Load test data.

In [ ]:
test_path = os.path.join(BASE_PATH, 'test.csv')
test_df = pd.read_csv(test_path)

test_df

# Processing "prompt", "response_a", and "response_b" 

Contents of each item looks list of strings. But, their type is just a string when reading from the csv files.

In [ ]:
first_prompt = train_df['prompt'].at[0]

print(f'1st prompt --> {first_prompt}')
print(f'its type --> {type(first_prompt)}')

To convert to list of strings, [eval](https://docs.python.org/3/library/functions.html#eval) can be used. It parses and evaluates the specificed argument as a Python expression, then returns the result. In this case, each itme in these columns are passed, and list of strings should be returned. But there are some problems. The first one is "NameError: name 'null' is not defined". 

In [ ]:
try:
    train_df['response_a'].map(lambda x: eval(x))
except Exception as e:
    print(f'Exception --> {e}')

Take a look the data which causes this error.

In [ ]:
for response_a_idx, response_a in enumerate(train_df['response_a']):
    try:
        eval(response_a)
    except Exception as e:
        print(f'Exception --> {e}')
        print(f'at --> {response_a_idx}')
        print(f'response_a --> {response_a}')
        break

"null" is used as an item in the list. It is definitely not defined and causes this error.
 
To fix the issue, the value for "null" should be defined when evaluating the strings. It's possible by specifying an additional argument to "eval". In the code below, '' (empty string) is defined for the value of "null" and all the items are successfully evaluated.

In [ ]:
null_value_dict = {'null': ''}
evaluated_response_a = \
    train_df['response_a'].map(lambda x: eval(x, {}, null_value_dict))

evaluated_response_a.at[response_a_idx]

The next issue is the characters which can not be represented as utf-8. Such characters cause errors when preprocessing to feed into language models. The code below shows an example of such an error.

In [ ]:
for prompt_idx, prompt in enumerate(train_df['prompt']):
    try:
        evaluated_prompt = eval(prompt, {}, null_value_dict)
        for raw_sentence in evaluated_prompt:
            encoded_prompt = raw_sentence.encode('utf-8')
    except Exception as e:
        prompt_with_error = prompt
        print(f'Exception --> {e}')
        print(f'at --> {prompt_idx}')
        print(f'prompt --> {prompt_with_error}')
        break

For the characters that cannot be represented as utf-8, let's ignore them. The function below converts each item in data. It encodes to a sequence of bytes, then decodes back to a string, while errors are ignored.

In [ ]:
def convert_data_item(raw_data):
    evaluated_data = eval(raw_data, {}, null_value_dict)
    converted_data = []
    for raw_str in evaluated_data:
        encoded_bytes = raw_str.encode('utf-8', errors='ignore')
        decoded_str = encoded_bytes.decode('utf-8', errors='ignore')
        converted_data.append(decoded_str)
    return converted_data

Convert the data, then take a look at an item which had caused the error. Unfortunately, characters which are not represented as utf-8 are removed, but the others remain the same.

In [ ]:
converted_prompt = train_df['prompt'].map(convert_data_item)

converted_prompt.at[prompt_idx]

Define functions to convert data and labels.

In [ ]:
def convert_data(df):
    df['converted_prompt'] = df['prompt'].map(convert_data_item)
    df['converted_response_a'] = df['response_a'].map(convert_data_item)
    df['converted_response_b'] = df['response_b'].map(convert_data_item)
    return df

def convert_label(df):
    df['class_name'] = df[
        ['winner_model_a', 'winner_model_b', 'winner_tie']].idxmax(axis=1)
    df['class_label'] = df['class_name'].map(CFG.name2label)
    return df

Convert training data and labels.

In [ ]:
train_df = convert_data(train_df)
train_df = convert_label(train_df)

train_df.head()

Convert test data.

In [ ]:
test_df = convert_data(test_df)

test_df.head()

# Data Analysis

Let's analyze sentenses and words in 'prompt', 'response_a', and 'response_b', which would be useful to make prompt and response pair. Length of these pairs are longer than sequence length of the used language model. It's important how to use this limited length for them. As the first step, make columns for the number of sentenses, words in each sentense, and total words.

In [ ]:
def make_word_count_list(converted_data):
    return [len(sentense.split()) for sentense in converted_data]

def make_count_data(
        data_col_name, len_col_name,
        word_count_list_col_name, total_word_count_col_name):
    train_df[len_col_name] = train_df[data_col_name].apply(len)
    train_df[word_count_list_col_name] = \
        train_df[data_col_name].apply(make_word_count_list)
    train_df[total_word_count_col_name] = \
        train_df[word_count_list_col_name].apply(sum)    

In [ ]:
make_count_data(
    'converted_prompt', 'prompt_len',
    'prompt_word_count_list', 'prompt_total_word_count')
make_count_data(
    'converted_response_a', 'response_a_len',
    'response_a_word_count_list', 'response_a_total_word_count')
make_count_data(
    'converted_response_b', 'response_b_len',
    'response_b_word_count_list', 'response_b_total_word_count')

train_df[[
    'prompt_len', 'prompt_total_word_count',
    'response_a_len', 'response_a_total_word_count',
    'response_b_len', 'response_b_total_word_count']].describe()

Take a look at the distribution and cumulative ratio for the prompt length. ~87% of prompt has only 1 sentense.

In [ ]:
prompt_len_bins = np.arange(1, 41)
prompt_len_counts, _, _ = plt.hist(train_df['prompt_len'], bins=prompt_len_bins)
plt.xlim(1, 40)
plt.ylim(0, 60000)
plt.title('Distribution of prompt length')
plt.xlabel('Prompt length')
plt.ylabel('Count')

prompt_len_cumulative_count = np.cumsum(prompt_len_counts)
prompt_len_cumulative_ratio = \
    prompt_len_cumulative_count / prompt_len_cumulative_count[-1]
ax2 = plt.gca().twinx()
ax2.plot(
    prompt_len_bins[1:], prompt_len_cumulative_ratio, color='orange')
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Cumulative ratio')

plt.show()

prompt_len_df = pd.DataFrame(
    {
        'Number of sentences': prompt_len_bins[:-1],
        'Count': prompt_len_counts,
        'Cumulative ratio': prompt_len_cumulative_ratio,
    }
)

prompt_len_df.head()

Next, let's check the distribution and cumulative ratio for the word count of the first sentence in the prompt. ~89% of first prompt sentences have 100 words or less.

In [ ]:
prompt_1st_sentence_word_count_bins = np.arange(1, 2501, 50)
prompt_1st_sentence_word_count = \
    train_df['prompt_word_count_list'].map(lambda x: x[0])
prompt_1st_sentence_word_dist, _, _ = \
    plt.hist(prompt_1st_sentence_word_count, bins=prompt_1st_sentence_word_count_bins)
plt.xlim(1, 2500)
plt.ylim(0, 50000)
plt.title('Distribution of word count for the first prompt sentense')
plt.xlabel('Word count for the first prompt sentense')
plt.ylabel('Count')

prompt_1st_sentence_word_cumulative_count = \
    np.cumsum(prompt_1st_sentence_word_dist)
prompt_1st_sentence_word_cumulative_ratio = \
    prompt_1st_sentence_word_cumulative_count \
    / prompt_1st_sentence_word_cumulative_count[-1]
ax2 = plt.gca().twinx()
ax2.plot(
    prompt_1st_sentence_word_count_bins[1:],
    prompt_1st_sentence_word_cumulative_ratio, color='orange')
ax2.set_ylim(0, 1.1)
ax2.set_ylabel('Cumulative ratio')

plt.show()

prompt_1st_sentence_word_count_df = pd.DataFrame({
    ('Number of words', 'from'): prompt_1st_sentence_word_count_bins[:-1],
    ('Number of words', 'to'): [i - 1 for i in prompt_1st_sentence_word_count_bins[1:]],
    ('', 'Count'): prompt_1st_sentence_word_dist,
    ('', 'Cumulative ratio'): prompt_1st_sentence_word_cumulative_ratio,
})

prompt_1st_sentence_word_count_df.head()

Make sure the numbers for response a and b. ~93% of them has 500 words or less.

In [ ]:
def plot_response_word_count(a_or_b, word_count, bins, subplot):
    plt.subplot(*subplot)
    plt.title(f'Response {a_or_b} word count distribution')
    response_word_dist, _, _ = plt.hist(word_count, bins=bins)
    plt.xlim(1, 3000)
    plt.ylim(0, 12000)
    plt.xlabel(f'Word count for resonse {a_or_b}')
    plt.ylabel('Count')

    response_word_cumulative_count = np.cumsum(response_word_dist)
    response_word_cumulative_ratio = \
        response_word_cumulative_count / response_word_cumulative_count[-1]
    ax2 = plt.gca().twinx()
    ax2.plot(bins[1:], response_word_cumulative_ratio, color='orange')
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel('Cumulative ratio')
    return response_word_dist, response_word_cumulative_ratio

response_word_count_bins = np.arange(1, 3001, 50)
plt.figure(figsize=(12, 4))
response_a_word_dist, response_a_word_cumulative_ratio = plot_response_word_count(
    'a', train_df['response_a_total_word_count'], response_word_count_bins, (1, 2, 1))
response_b_word_dist, response_b_word_cumulative_ratio = plot_response_word_count(
    'b', train_df['response_b_total_word_count'], response_word_count_bins, (1, 2, 2))
plt.subplots_adjust(wspace=0.4)
plt.show()

In [ ]:
response_a_word_count_df = pd.DataFrame({
    ('Number of words', 'from'): response_word_count_bins[:-1],
    ('Number of words', 'to'): [i - 1 for i in response_word_count_bins[1:]],
    ('', 'Count'): response_a_word_dist,
    ('', 'Cumulative ratio'): response_a_word_cumulative_ratio,
})

response_a_word_count_df.head(10)

In [ ]:
response_b_word_count_df = pd.DataFrame({
    ('Number of words', 'from'): response_word_count_bins[:-1],
    ('Number of words', 'to'): [i - 1 for i in response_word_count_bins[1:]],
    ('', 'Count'): response_b_word_dist,
    ('', 'Cumulative ratio'): response_b_word_cumulative_ratio,
})

response_b_word_count_df.head(10)

## Contextualize Response with Prompt

According to the analysis, at most 100 words are used for the prompts and all the rest are used for responses. The sequence length of the used language model is 512.

In [ ]:
def make_pairs(row):
    # For the prompts, at most 100 words are used.
    all_prompt_sentences = " ".join(row['converted_prompt'])
    prompt_beginning_100_words = all_prompt_sentences.split()[:100]
    used_prompt = " ".join(prompt_beginning_100_words)

    # For the responses, all items are used, but it's limited to the sequence length
    # of the language model.
    response_a = " ".join(row['converted_response_a'])
    response_b = " ".join(row['converted_response_b'])

    # Make options
    row['options'] = [
        f'Prompt: {used_prompt}\n\nResponse: {response_a}',
        f'Prompt: {used_prompt}\n\nResponse: {response_b}',
    ]
    return row

In [ ]:
train_df = train_df.apply(make_pairs, axis=1)

train_df.head()

In [ ]:
test_df = test_df.apply(make_pairs, axis=1)

test_df.head()

# Data Split

Divide the provided data into training and validation using a stratification of class_label column.

In [ ]:
from sklearn.model_selection import train_test_split

split_train_df, split_valid_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df['class_label'])

Make sure the ratio for each label is maintained in the split data.

In [ ]:
def calc_class_label_count(df):
    return df['class_label'].value_counts()

def calc_class_label_ratios(df):
    col = calc_class_label_count(df)
    total = col.sum()
    ratios = col.apply(lambda x: x / total * 100)
    return ratios

class_label_counts = pd.DataFrame({
    ('Train', 'Count'): calc_class_label_count(train_df),
    ('Train', 'Ratio'): calc_class_label_ratios(train_df),
    ('Split Train', 'Count'): calc_class_label_count(split_train_df),
    ('Split Train', 'Ratio'): calc_class_label_ratios(split_train_df),
    ('Split Valid', 'Count'): calc_class_label_count(split_valid_df),
    ('Split Valid', 'Ratio'): calc_class_label_ratios(split_valid_df)
})

class_label_counts

# Preprocessing

In [ ]:
preprocessor = CFG.preprocessor.from_preset(
    preset=CFG.preset,
    sequence_length=CFG.sequence_length
)

preprocessor

In [ ]:
output = preprocessor(train_df['options'].iloc[0])

for k, v in output.items():
    print(k, ":", v.shape)

In [ ]:
def preprocess_fn(text, label=None):
    tokens = preprocessor(text)
    return (tokens, label) if label is not None else (tokens, 0)

# Data Loader

In [ ]:
def build_dataset(texts, labels=None, batch_size=32, cache=True, shuffle=1024):
    AUTO = tf.data.AUTOTUNE
    slices = (texts, ) if labels is None \
        else (texts, keras.utils.to_categorical(labels, num_classes=3))
    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(preprocess_fn, num_parallel_calls=AUTO)
    opt = tf.data.Options()
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt.experimental_deterministic = False
    ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTO)
    return ds

When debugging, reduce the data to speed up the process.

In [ ]:
split_train_df = split_train_df.sample(frac=0.05) if DEBUG else split_train_df
split_valid_df = split_valid_df.sample(frac=0.05) if DEBUG else split_valid_df

In [ ]:
train_texts = split_train_df.options.tolist()
train_labels = split_train_df.class_label.tolist()
train_ds = build_dataset(
    train_texts, train_labels, batch_size=CFG.batch_size, shuffle=True)

train_ds

In [ ]:
valid_texts = split_valid_df.options.tolist()
valid_labels = split_valid_df.class_label.tolist()
valid_ds = build_dataset(
    valid_texts, valid_labels, batch_size=CFG.batch_size, shuffle=False)

valid_ds

# Learning Rate Schedule

In [ ]:
import math

def get_lr_callback(batch_size=8, epochs=10, plot=False):
    lr_start, lr_max, lr_min = 1e-6, 0.3e-6 * batch_size, 1e-6
    lr_ramp_ep, lr_sus_ep, lr_decay = 2, 0, 0.8

    def lrfn(epoch):
        if epoch < lr_ramp_ep:
            lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep:
            lr = lr_max
        else:
            decay_total_epochs = epochs - (lr_ramp_ep + lr_sus_ep) - 1
            decay_epoch_index = epoch - (lr_ramp_ep + lr_sus_ep)
            phase = math.pi * decay_epoch_index / decay_total_epochs
            # print(decay_total_epochs, decay_epoch_index, phase)
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:
        plt.figure()
        x_values = np.arange(1, epochs + 1)
        lr_values = [lrfn(epoch) for epoch in range(epochs)]
        plt.plot(x_values, lr_values, marker='o')
        plt.xlabel('epoch')
        plt.ylabel('lr')
        plt.ylim(0, 0.6e-5)
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=True)

lr_cb = get_lr_callback(CFG.batch_size, CFG.epochs, plot=True)

# Model Checkpointing and Early Stopping

In [ ]:
best_model_file_name = 'best_model.weights.h5'

ckpt_cb = keras.callbacks.ModelCheckpoint(
    best_model_file_name, monitor='val_loss', save_best_only=True,
    save_weights_only=True, mode='min')

early_stopping_cb = keras.callbacks.EarlyStopping(
    monitor='val_loss', min_delta=0, patience=1, verbose=1,
    mode='min', baseline=None, restore_best_weights=False, start_from_epoch=0
)

# Model

After obtaining embeddings for all responses, concatenate them and take the first dimension for this classification task.

In [ ]:
inputs = {
    "token_ids": keras.Input(shape=(2, None), dtype=tf.int32, name="token_ids"),
    "padding_mask": keras.Input(shape=(2, None), dtype=tf.int32, name="padding_mask")
}

backbone = CFG.backbone.from_preset(CFG.preset)

response_a = {k: v[:, 0, :] for k, v in inputs.items()}
embed_a = backbone(response_a)

response_b = {k: v[:, 1, :] for k, v in inputs.items()}
embed_b = backbone(response_b)

x = keras.layers.Concatenate(axis=-1)([embed_a, embed_b])
x = x[:, 0, :]
outputs = keras.layers.Dense(3, activation='softmax', name='classifier')(x)

model = keras.Model(inputs, outputs)

model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(5e-6),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    metrics=[
        keras.metrics.CategoricalAccuracy(name="accuracy")
    ]
)

# Training

In [ ]:
history = model.fit(
    train_ds,
    epochs=CFG.epochs,
    validation_data=valid_ds,
    callbacks=[lr_cb, ckpt_cb, early_stopping_cb]
)

Check how loss and accuracy is improved in the training.

In [ ]:
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.title('Accuracy')
plt.legend()
plt.show()

## Load Best Model

In [ ]:
best_model_file_path = os.path.join(
    '/', 'kaggle', 'working', best_model_file_name)
model.load_weights(best_model_file_path)

# Prediction

In [ ]:
test_texts = test_df.options.tolist()
test_batch_size = min(len(test_df), CFG.batch_size)
test_ds = build_dataset(
    test_texts, labels=None, batch_size=test_batch_size, shuffle=False)

test_ds

In [ ]:
test_preds = model.predict(test_ds, verbose=1)

# Submission

In [ ]:
sub_df = test_df[['id']].copy()
sub_df[CFG.class_names] = test_preds.tolist()
sub_df.to_csv('submission.csv', index=False)

sub_df.head()